# Fuzz Archives: Artist-Centered Graph (Hardcore Filter)

*Powered by **Traverse** and Visualized with **Cosmograph***

Same as the artist-centered graph, but filtered to **only** include artists
whose styles include **Hardcore** or **Happy Hardcore**.  Uses the
`require_tags` parameter to pre-filter nodes before edge building.

In [ ]:
# Ensure we import traverse from traverse_vc (not the old traverse repo)
import sys
from pathlib import Path
_vc_src = str(Path(r"C:\Users\xtrem\Documents\Projects\GitHub\traverse_vc\src"))
if _vc_src not in sys.path:
    sys.path.insert(0, _vc_src)
# Force reload in case old modules are cached
for mod_name in [k for k in sys.modules if k.startswith("traverse")]:
    del sys.modules[mod_name]

## 1. Configuration

In [10]:
from pathlib import Path

RECORDS_CSV = Path(r"C:\Users\xtrem\Documents\Datasets\records.csv")
OUT_DIR = Path("_out_artist_happyhardcore")
FORCE = True  # set True to rebuild cache

# Tag filter — keep only artists with these styles
REQUIRE_TAGS = {"styles": ["Happy Hardcore"]}

# Tuning parameters
MAX_NODES = 0               # 0 = unlimited
MIN_SHARED_TAGS = 2         # min shared tags to create an edge
MAX_EDGES = 0               # 0 = unlimited
MAX_EDGES_PER_NODE = 2      # keep only top-K edges per artist
MAX_TAG_DEGREE = 200       # sample tags shared by more artists than this
SAMPLE_HIGH_DEGREE = True   # True = sample down; False = skip entirely

# When True and multiple tag_types are used, require shared tags in
# EVERY tag type (AND logic) instead of ANY tag type (OR logic).
REQUIRE_ALL_TAG_TYPES = True

## 2. Build or Load Graph

In [11]:
from traverse.graph.artist_graph import build_artist_graph
from traverse.graph.cache import GraphCache

cache = GraphCache(
    cache_dir=OUT_DIR,
    build_fn=lambda: build_artist_graph(
        RECORDS_CSV,
        max_nodes=MAX_NODES,
        min_shared_tags=MIN_SHARED_TAGS,
        max_edges=MAX_EDGES,
        max_edges_per_node=MAX_EDGES_PER_NODE,
        max_tag_degree=MAX_TAG_DEGREE,
        sample_high_degree=SAMPLE_HIGH_DEGREE,
        require_tags=REQUIRE_TAGS,
        require_all_tag_types=REQUIRE_ALL_TAG_TYPES,
    ),
    force=FORCE,
)
graph, records_df = cache.load_or_build()

n_pts = len(graph["points"])
n_lks = len(graph["links"])
total_items = n_pts + n_lks
print(f"Graph: {n_pts:,} nodes, {n_lks:,} edges ({total_items:,} total items)")
print(f"Records: {len(records_df):,} rows")

Building graph (this may take a while)…
Reading records: 74chunk [11:38,  9.44s/chunk]
Pass 1a: 14,609,075 rows scanned, 2,851,681 unique artists with tags (edge tag_types=['styles'])
  require_tags filter: 2,851,681 → 11,198 artists (require={'styles': ['Happy Hardcore']})
Finding neighbors: 100%|██████████| 11198/11198 [00:02<00:00, 4013.62node/s]
Pass 2 (nearest-neighbor): 19,644 unique edges, 37 tags sampled, 0 tags skipped (degree > 200)
Pass 3: 9,904 artist nodes, 19,644 edges (min_shared_tags=2, max_edges=0, max_edges_per_node=2)


Graph: 9,904 nodes, 19,644 edges (29,548 total items)
Records: 9,904 rows


Cached graph → _out_artist_happyhardcore\graph.json
Cached records → _out_artist_happyhardcore\canonical_plays.parquet (9,904 rows)


## 3. Community Detection

In [12]:
from collections import Counter
from traverse.graph.community import add_communities, CommunityAlgorithm

graph = add_communities(graph, CommunityAlgorithm.LOUVAIN, seed=42)

comm_counts = Counter(pt["community"] for pt in graph["points"])
print(f"{len(comm_counts)} communities:")
for comm_id, count in comm_counts.most_common():
    print(f"  Community {comm_id}: {count} nodes")

15 communities:
  Community 0: 3727 nodes
  Community 1: 1085 nodes
  Community 2: 825 nodes
  Community 3: 822 nodes
  Community 4: 594 nodes
  Community 5: 500 nodes
  Community 6: 453 nodes
  Community 7: 448 nodes
  Community 8: 375 nodes
  Community 9: 324 nodes
  Community 10: 261 nodes
  Community 11: 184 nodes
  Community 12: 122 nodes
  Community 13: 97 nodes
  Community 14: 87 nodes


## 3b. User Listening Overlap

Overlay your Spotify listening history to see which artists you've actually
listened to.  Matched nodes get `first_seen_ts` (for timeline) and
`play_count` fields injected into the graph.

In [ ]:
from traverse.graph.user_overlap import compute_user_overlap

SPOTIFY_HISTORY = Path(r"..\Spotify\anthony\ExtendedStreamingHistory")

overlap = compute_user_overlap(graph, history_dir=SPOTIFY_HISTORY)
print(f"Matched {overlap['totalMatched']} of {overlap['totalNodes']} nodes "
      f"({overlap['totalMatched'] / overlap['totalNodes'] * 100:.1f}%)")

# Inject first_seen_ts + play_count onto graph points for timeline & export
overlap_map = {m["nodeId"]: m for m in overlap["matches"]}
for pt in graph["points"]:
    match = overlap_map.get(pt["id"])
    if match:
        pt["first_seen_ts"] = match.get("firstListenEpochMs")
        pt["play_count"] = match["playCount"]

# Show top 15 most-played matches
print("\nTop 15 most-played artists in this graph:")
for m in overlap["matches"][:15]:
    mins = m["totalMs"] // 60_000
    top = m["topTracks"][0]["trackName"] if m["topTracks"] else "—"
    print(f"  {m['playCount']:>4}x  {mins:>5}m  {m['nodeId']}  (top: {top})")

## 4. Sanity Check

In [13]:
import random

sample_pt = random.choice(graph["points"])
sample_id = sample_pt["id"]
print(f"Sample node: {sample_pt}")
print()

# Find neighbors
id_to_pt = {pt["id"]: pt for pt in graph["points"]}
neighbors = []
for lk in graph["links"]:
    w = lk.get("weight", 1)
    if lk["source"] == sample_id:
        neighbors.append((lk["target"], w))
    elif lk["target"] == sample_id:
        neighbors.append((lk["source"], w))

neighbors.sort(key=lambda x: x[1], reverse=True)
print(f"{len(neighbors)} neighbors (top 10 by shared tags):")
for nid, w in neighbors[:10]:
    npt = id_to_pt.get(nid, {})
    print(f"  w={w}: {npt.get('label', nid)}")

Sample node: {'id': 'Pineapple Jack', 'label': 'Pineapple Jack', 'genres': 'Electronic', 'styles': 'Gabber | Happy Hardcore | Hardcore', 'external_links': [{'platform': 'discogs', 'url': 'https://www.discogs.com/search/?q=Pineapple+Jack&type=artist', 'label': 'Discogs'}], 'community': 7}

2 neighbors (top 10 by shared tags):
  w=2: Charly Lownoise & Mental Theo
  w=2: Abitan


## 5. Export & Serve

In [ ]:
from traverse.graph.adapters_cosmograph import CosmographAdapter
from traverse.cosmograph.server import serve, _default_dist_dir

n_pts = len(graph["points"])
n_lks = len(graph["links"])
print(f"Graph: {n_pts:,} nodes, {n_lks:,} edges")

if n_lks > 500_000:
    print(f"WARNING: {n_lks:,} edges is too many for browser visualization.")
    print("Consider increasing MIN_SHARED_TAGS or lowering MAX_EDGES/MAX_NODES.")
    print("Skipping export. Adjust params and re-run.")
else:
    meta = {"clusterField": "community", "title": "Fuzz Archives — Hardcore"}
    out_path = _default_dist_dir() / "cosmo_artists_happyhardcore.json"
    CosmographAdapter.write(graph, out_path, meta=meta)
    print()
    print("Starting server — open in browser:")
    print("  Local:   http://127.0.0.1:8080/?data=/cosmo_artists_happyhardcore.json")
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        local_ip = s.getsockname()[0]
        s.close()
        print(f"  Network: http://{local_ip}:8080/?data=/cosmo_artists_happyhardcore.json")
    except Exception:
        print("  Network: http://<your-local-ip>:8080/?data=/cosmo_artists_happyhardcore.json")
    print()
    print("Press Ctrl+C (or interrupt the kernel) to stop.")
    serve(port=8080, host="0.0.0.0")